# 🥉 Bronze Layer - Sales Data Ingestion
## AWS S3 External Storage via Unity Catalog

**Purpose**: Ingest Sales schema tables from source to Bronze layer

**Source**: `adventureworks_live_conn_catalog.sales_lt.*`
**Target**: `workspace.`workspace-adventureworks-bronze`` (S3)
**Tables**: 19 tables (~250K rows)

**Key Tables**:
- Customer
- SalesOrderHeader
- SalesOrderDetail
- Address
- CustomerAddress
- Product
- ProductCategory
- ProductModel
- ProductDescription

**Transformation**: Add audit columns (ingestion_timestamp, source_system)

In [0]:
from pyspark.sql.functions import current_timestamp, lit
from datetime import datetime
import time

# Configuration
SOURCE_CATALOG = "adventureworks_live_conn_catalog"
SOURCE_SCHEMA = "sales"
TARGET_SCHEMA = "workspace.`workspace-adventureworks-bronze`"
SOURCE_SYSTEM = "AdventureWorks_OLTP"

print("=" * 80)
print("🥉 BRONZE LAYER - SALES DATA INGESTION")
print("=" * 80)
print(f"Start Time: {datetime.now()}")
print(f"Source: {SOURCE_CATALOG}.{SOURCE_SCHEMA}")
print(f"Target: {TARGET_SCHEMA} (AWS S3 External Storage)")
print(f"Source System: {SOURCE_SYSTEM}")
print("=" * 80)
print()

# Track ingestion results
ingestion_results = []

In [0]:
def ingest_table_to_bronze(source_table_name, target_table_name=None):
    """
    Ingest a single table from source to Bronze with audit columns
    
    Args:
        source_table_name: Name of the source table
        target_table_name: Optional custom target name (defaults to source name)
    
    Returns:
        dict: Ingestion result with metrics
    """
    if target_table_name is None:
        target_table_name = source_table_name
    
    source_full = f"{SOURCE_CATALOG}.{SOURCE_SCHEMA}.{source_table_name}"
    target_full = f"{TARGET_SCHEMA}.{target_table_name}"
    
    result = {
        "source_table": source_table_name,
        "target_table": target_table_name,
        "status": "failed",
        "row_count": 0,
        "duration_seconds": 0,
        "error": None
    }
    
    try:
        start_time = time.time()
        print(f"\n⏳ Ingesting: {source_table_name}...", end=" ")
        
        # Read source table
        df = spark.table(source_full)
        print(f"Read {df.count()} rows")
        # Add audit columns
        df_bronze = (df
            .withColumn("ingestion_timestamp", current_timestamp())
            .withColumn("source_system", lit(SOURCE_SYSTEM))
        )
        
        # Write to Bronze (S3) with overwrite mode
        (df_bronze
            .write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(target_full)
        )
        
        # Get row count
        row_count = df_bronze.count()
        duration = time.time() - start_time
        
        result.update({
            "status": "success",
            "row_count": row_count,
            "duration_seconds": round(duration, 2)
        })
        
        print(f"✅ {row_count:,} rows in {duration:.2f}s")
        
    except Exception as e:
        duration = time.time() - start_time
        result.update({
            "error": str(e),
            "duration_seconds": round(duration, 2)
        })
        print(f"❌ Failed: {str(e)}")
    
    return result

In [0]:
# Get list of tables in Sales schema
print("🔍 Discovering Sales tables...")

try:
    sales_tables = spark.sql(f"SHOW TABLES IN {SOURCE_CATALOG}.{SOURCE_SCHEMA}").collect()
    table_names = [row.tableName for row in sales_tables]
    print(f"✅ Found {len(table_names)} tables in Sales schema")
    print("\nTables to ingest:")
    for i, table in enumerate(table_names, 1):
        print(f"  {i}. {table}")
except Exception as e:
    print(f"❌ Error discovering tables: {str(e)}")
    raise

In [0]:
# Ingest all Sales tables to Bronze
print("\n" + "=" * 80)
print("🚀 STARTING INGESTION")
print("=" * 80)

overall_start = time.time()

for table_name in table_names:
    result = ingest_table_to_bronze(table_name)
    ingestion_results.append(result)

overall_duration = time.time() - overall_start

print("\n" + "=" * 80)
print("🏁 INGESTION COMPLETE")
print("=" * 80)
print(f"Total Duration: {overall_duration:.2f}s ({overall_duration/60:.2f} minutes)")

In [0]:
# Display summary
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

print("\n📊 INGESTION SUMMARY\n")

success_count = sum(1 for r in ingestion_results if r["status"] == "success")
failed_count = sum(1 for r in ingestion_results if r["status"] == "failed")
total_rows = sum(r["row_count"] for r in ingestion_results)

print(f"Tables Processed: {len(ingestion_results)}")
print(f"  ✅ Success: {success_count}")
print(f"  ❌ Failed: {failed_count}")
print(f"Total Rows Ingested: {total_rows:,}")

# Create summary DataFrame with explicit schema
schema = StructType([
    StructField("source_table", StringType(), False),
    StructField("target_table", StringType(), False),
    StructField("status", StringType(), False),
    StructField("row_count", IntegerType(), False),
    StructField("duration_seconds", DoubleType(), False),
    StructField("error", StringType(), True)
])

summary_data = [
    Row(
        source_table=r["source_table"],
        target_table=r["target_table"],
        status=r["status"],
        row_count=r["row_count"],
        duration_seconds=r["duration_seconds"],
        error=r["error"]
    )
    for r in ingestion_results
]

summary_df = spark.createDataFrame(summary_data, schema=schema)
print("\nDetailed Results:")
display(summary_df.orderBy("status", "source_table"))

# Show failed tables if any
if failed_count > 0:
    print("\n⚠️ FAILED TABLES:")
    for r in ingestion_results:
        if r["status"] == "failed":
            print(f"  - {r['source_table']}: {r['error']}")

In [0]:
# Verify tables exist in Bronze schema
print("\n" + "=" * 80)
print("🔍 VERIFYING BRONZE TABLES")
print("=" * 80)

try:
    bronze_tables = spark.sql(f"SHOW TABLES IN {TARGET_SCHEMA}").collect()
    bronze_table_names = [row.tableName for row in bronze_tables]
    
    print(f"\n✅ Found {len(bronze_table_names)} tables in Bronze schema")
    print(f"   Location: AWS S3 (Unity Catalog External Storage)")
    print(f"   Format: Delta Lake")
    
    # Sample one table to verify audit columns
    if bronze_table_names:
        sample_table = bronze_table_names[0]
        sample_df = spark.table(f"{TARGET_SCHEMA}.{sample_table}").limit(1)
        columns = sample_df.columns
        
        if "ingestion_timestamp" in columns and "source_system" in columns:
            print(f"\n✅ Audit columns verified: ingestion_timestamp, source_system")
        else:
            print(f"\n⚠️ Warning: Audit columns not found in {sample_table}")
            
except Exception as e:
    print(f"\n❌ Error verifying Bronze tables: {str(e)}")

In [0]:
# Return results for orchestration
print("\n" + "=" * 80)
if failed_count == 0:
    print("✅ BRONZE SALES INGESTION COMPLETED SUCCESSFULLY")
    print("=" * 80)
    print(f"\n✓ {success_count} tables ingested to S3-backed Bronze schema")
    print(f"✓ {total_rows:,} total rows")
    print(f"\nCompletion Time: {datetime.now()}")
    dbutils.notebook.exit("SUCCESS")
else:
    print("⚠️ BRONZE SALES INGESTION COMPLETED WITH ERRORS")
    print("=" * 80)
    print(f"\n✓ Success: {success_count} tables")
    print(f"❌ Failed: {failed_count} tables")
    print(f"\nCompletion Time: {datetime.now()}")
    dbutils.notebook.exit(f"PARTIAL_SUCCESS: {failed_count} tables failed")